# Scraping El Comercio - Noticias

**Objetivo:** Extraer 1000 artículos de noticias para trabajo de NLP

**URL listado:** https://elcomercio.pe/ultimas-noticias/

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import random
import pandas as pd

URL_LISTADO = 'https://elcomercio.pe/ultimas-noticias/'
TOTAL_ARTICULOS = 1000
DELAY_MIN = 5
DELAY_MAX = 10
REINTENTOS_MAX = 5

In [ ]:
options = Options()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
options.add_argument('--lang=es')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

In [ ]:
def extraer_listado(driver):
    articulos = []
    headings = driver.find_elements(By.TAG_NAME, 'h2')
    
    for h in headings:
        try:
            link = h.find_element(By.TAG_NAME, 'a')
            url = link.get_attribute('href')
            titulo = link.text.strip()
            if url and titulo and 'elcomercio.pe' in url:
                articulos.append({'titulo': titulo, 'url': url})
        except:
            continue
    
    return articulos

In [ ]:
def extraer_articulo(driver, url):
    try:
        driver.get(url)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
        time.sleep(3)
        
        try:
            overlay = driver.find_element(By.ID, 'adk_closing-x')
            overlay.click()
            time.sleep(1)
        except:
            pass
        
        try:
            titulo = driver.find_element(By.TAG_NAME, 'h1').text.strip()
        except:
            titulo = None
        
        try:
            contenido = driver.find_element(By.CSS_SELECTOR, '[itemprop="articleBody"], div.article-body, div[class*="article-body"]').text.strip()
        except:
            try:
                contenido = driver.find_element(By.TAG_NAME, 'article').text.strip()
            except:
                contenido = None
        
        try:
            fecha = driver.find_element(By.CSS_SELECTOR, 'time, span[class*="date"], div[class*="date"]').text.strip()
        except:
            fecha = None
        
        try:
            autor = driver.find_element(By.CSS_SELECTOR, 'a[href*="/autor/"], span[class*="author"], div[class*="author"]').text.strip()
        except:
            autor = None
        
        return {
            'titulo': titulo,
            'contenido': contenido,
            'fecha_publicacion': fecha,
            'autor': autor,
            'url': url
        }
    except Exception as e:
        print(f'Error extrayendo artículo {url}: {e}')
        return None

In [ ]:
driver.get(URL_LISTADO)
time.sleep(5)

datos = []
urls_procesadas = set()
errores_consecutivos = 0

while len(datos) < TOTAL_ARTICULOS:
    print(f'Artículos: {len(datos)} | Procesando listado...')
    
    try:
        articulos_listado = extraer_listado(driver)
        
        for articulo in articulos_listado:
            if len(datos) >= TOTAL_ARTICULOS:
                break
            
            url = articulo['url']
            if url in urls_procesadas:
                continue
            urls_procesadas.add(url)
            
            resultado = extraer_articulo(driver, url)
            if resultado and resultado['contenido'] and len(resultado['contenido']) > 50:
                datos.append(resultado)
                print(f'  Extraído: {resultado["titulo"][:60]}... | Total: {len(datos)}')
            
            if len(datos) % 50 == 0 and len(datos) > 0:
                df_temp = pd.DataFrame(datos)
                df_temp.to_csv('noticias_elcomercio.csv', index=False, encoding='utf-8-sig')
                print(f'Checkpoint guardado: {len(datos)} artículos')
            
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        
        errores_consecutivos = 0
        
        try:
            btn_cargar_mas = driver.find_element(By.XPATH, '//button[contains(text(),"Cargar más") or contains(text(),"Ver más") or contains(@class,"load-more")]')
            btn_cargar_mas.click()
            time.sleep(random.uniform(8, 12))
        except:
            print('No se encontró botón de cargar más. Fin del scraping.')
            break
        
    except Exception as e:
        errores_consecutivos += 1
        print(f'Error en listado: {e}')
        if errores_consecutivos >= REINTENTOS_MAX:
            print('Demasiados errores consecutivos. Pausa de 60s.')
            time.sleep(60)
            errores_consecutivos = 0
        else:
            time.sleep(10)

driver.quit()
print(f'Scraping finalizado. Total: {len(datos)} artículos')

In [ ]:
df = pd.DataFrame(datos)
df.to_csv('noticias_elcomercio.csv', index=False, encoding='utf-8-sig')
print(f'DataFrame final: {len(df)} artículos')
df.head()